# AIMO Progress Prize 3 — Kaggle Submission

**Strategy:** TIR (Tool-Integrated Reasoning) + Majority Voting  
**Model:** GPT-OSS-120B (OpenAI, MoE 117B total / 5.1B active, MXFP4 quantized)  

### Kaggle Setup:
1. Add model: `danielhanchen/gpt-oss-120b` (Transformers / default)
2. Add dataset: `sonphamorg/vllm-wheels-py312-cu129` (vLLM 0.15.0 offline wheels)
3. Enable GPU (H100)
4. Disable Internet
5. Submit

## Cell 1: Environment Setup

In [ ]:
import os
import sys
import re
import gc
import time
import tempfile
import subprocess
import warnings
from typing import Optional
from collections import Counter, defaultdict

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"

# NVIDIA-specific (Kaggle H100)
if os.path.exists("/usr/local/cuda/bin/ptxas"):
    os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"

# gpt-oss tiktoken encodings (if available from pip_install dataset)
_tiktoken_paths = [
    "/kaggle/usr/lib/pip_install_aimo3_1/tiktoken_encodings",
    "/kaggle/input/vllm-wheels-py312-cu129/tiktoken_encodings",
]
for _tp in _tiktoken_paths:
    if os.path.exists(_tp):
        os.environ["TIKTOKEN_ENCODINGS_BASE"] = _tp
        break

warnings.simplefilter("ignore")
print("Environment ready.")

## Cell 2: Imports

In [ ]:
import pandas as pd
import polars as pl

try:
    import kaggle_evaluation.aimo_3_inference_server
    HAS_KAGGLE_EVAL = True
    print("kaggle_evaluation available")
except ImportError:
    HAS_KAGGLE_EVAL = False
    print("kaggle_evaluation NOT available (local mode)")

## Cell 3: Configuration

In [ ]:
# --- Model ---
MODEL_PATH = None  # Auto-discovered below
MODEL_NAME = "GPT-OSS-120B"

# --- Inference ---
MAX_TOKENS = 32768          # Max tokens per generation (thinking + answer)
MAX_ROUNDS = 1              # Max TIR rounds (gpt-oss reasoning is strong, 1 round usually enough)
N_PROMPTS = 10              # Number of diverse prompts (= number of samples)
TEMPERATURE = 0.6
MIN_P = 0.05
SEED = 42
CODE_TIMEOUT = 10           # Seconds per code execution

# --- Time management ---
CUTOFF_TIME = time.time() + (4 * 60 + 45) * 60  # 4h45m (conservative for 9h limit, matching top notebooks)

# --- System prompts (diverse for majority voting) ---
SYSTEM_PROMPTS = [
    "Let's think step by step. Show your reasoning, then put the answer in \\boxed{}.",
    "Solve the problem step by step, reflect on your solution, and give the final answer in \\boxed{}.",
    "Work through the problem clearly, verify your reasoning, and provide the final answer in \\boxed{}.",
    "Explain your chain of thought, double-check your work, and then write the final answer in \\boxed{}.",
    "Think carefully, reason step-by-step, then confirm your answer and put it in \\boxed{}.",
    "Break down the problem logically, ensure each step is correct, and give the final answer in \\boxed{}.",
    "Reason carefully from start to finish, check the conclusion, and place the final answer in \\boxed{}.",
    "Analyze the problem step by step, validate your steps, and present the final answer in \\boxed{}.",
    "Think through the problem methodically, verify your solution, and write the final answer in \\boxed{}.",
    "Derive the solution one step at a time, confirm your result is correct, and put the final answer in \\boxed{}.",
]

print(f"Config: {N_PROMPTS} prompts, {MAX_ROUNDS} rounds, temp={TEMPERATURE}")
print(f"Cutoff: {(CUTOFF_TIME - time.time()) / 3600:.1f}h from now")

## Cell 4: Find Model Path

In [ ]:
def find_model_path():
    """Auto-discover model path from Kaggle inputs."""
    candidates = [
        "/kaggle/input/gpt-oss-120b/transformers/default/1",
        "/kaggle/input/gpt-oss-20b/transformers/default/1",
        "/kaggle/input/qwen3-30b-a3b-thinking-2507/transformers/default/1",
        "/kaggle/input/qwen3-30b-a3b/transformers/default/1",
        "/kaggle/input/qwen-3/transformers/30b-a3b/1",
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, "config.json")):
            return path
    # Walk /kaggle/input to find any model
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "config.json" in files and "tokenizer.json" in files:
                return root
    return None


MODEL_PATH = find_model_path()
if MODEL_PATH:
    print(f"Found model: {MODEL_PATH}")
else:
    print("No model found at /kaggle/input. This is expected locally.")
    print("On Kaggle, add a model input to your notebook.")

## Cell 5: Code Execution (subprocess-based)

In [ ]:
class PythonREPL:
    """Execute Python code in a subprocess with timeout."""
    def __init__(self, timeout=CODE_TIMEOUT):
        self.timeout = timeout

    def __call__(self, code: str) -> tuple:
        full_code = (
            "import math, numpy as np, sympy as sp, mpmath, itertools, collections\n"
            "from sympy import *\n"
            "mpmath.mp.dps = 64\n"
        ) + code
        with tempfile.TemporaryDirectory() as td:
            path = os.path.join(td, "run.py")
            with open(path, "w") as f:
                f.write(full_code)
            try:
                result = subprocess.run(
                    [sys.executable, path],
                    capture_output=True, text=True, timeout=self.timeout,
                )
            except subprocess.TimeoutExpired:
                return False, f"Timed out after {self.timeout}s"
            if result.returncode == 0:
                return True, result.stdout.strip()
            return False, result.stderr.strip()[-500:]

repl = PythonREPL()

# Quick test
ok, out = repl("print(2 + 2)")
print(f"REPL test: ok={ok}, output='{out}'")

## Cell 6: Answer Extraction

In [ ]:
def extract_boxed_answers(text: str) -> list:
    """Extract integer answers from \\boxed{...} patterns."""
    answers = []
    for needle in [r"\boxed{", "boxed{"]:
        i = 0
        while True:
            j = text.find(needle, i)
            if j < 0:
                break
            k = j + len(needle)
            depth = 1
            buf = []
            while k < len(text) and depth > 0:
                ch = text[k]
                if ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        break
                buf.append(ch)
                k += 1
            payload = "".join(buf).replace(",", "").replace("_", "").strip()
            for num_str in re.findall(r"\b\d{1,5}\b", payload):
                n = int(num_str)
                if 0 <= n <= 99999:
                    answers.append(n)
            i = max(k, j + 1)
    return answers


def extract_python_code(text: str) -> list:
    """Extract ```python ... ``` code blocks."""
    return re.findall(r"```python\s*\n?(.*?)```", text, re.DOTALL)


def strip_think(text: str) -> str:
    """Remove <think>...</think> blocks."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def select_answer(answers: list) -> int:
    """Majority vote over valid answers."""
    valid = [int(a) for a in answers if 0 <= int(a) <= 99999]
    if not valid:
        return 0
    return Counter(valid).most_common(1)[0][0]


# Quick test
test_answers = extract_boxed_answers(r"The answer is \boxed{42}.")
print(f"Boxed extraction test: {test_answers}")
print(f"Majority vote test: {select_answer([42, 42, 7, 42, 7])}")

## Cell 7: Model Wrapper

In [ ]:
class Model:
    """Wraps vLLM server (Kaggle) or external OpenAI-compatible API (local)."""

    def __init__(self):
        self._client = None
        self._model_name = None
        self._server_process = None
        self._log_file = None

    def _preload_weights(self, model_path):
        """Pre-read model files into OS page cache for faster server startup."""
        from concurrent.futures import ThreadPoolExecutor
        files = []
        for root, _, names in os.walk(model_path):
            for name in names:
                fp = os.path.join(root, name)
                if os.path.isfile(fp):
                    files.append(fp)
        def _read(path):
            with open(path, 'rb') as f:
                while f.read(1024 * 1024 * 1024):
                    pass
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=16) as pool:
            list(pool.map(_read, files))
        total_gb = sum(os.path.getsize(f) for f in files) / 1e9
        print(f"  Cached {len(files)} files ({total_gb:.1f} GB) in {time.time()-t0:.1f}s")

    def load_vllm_server(self, model_path, port=8000):
        """Start vLLM as a separate server process (avoids CUDA memory conflicts)."""
        print(f"Pre-loading model weights into page cache...")
        self._preload_weights(model_path)

        print(f"Starting vLLM server on port {port}...")
        cmd = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed', str(SEED),
            '--model', model_path,
            '--served-model-name', 'gpt-oss',
            '--tensor-parallel-size', '1',
            '--max-num-seqs', '128',
            '--gpu-memory-utilization', '0.96',
            '--host', '0.0.0.0',
            '--port', str(port),
            '--dtype', 'auto',
            '--kv-cache-dtype', 'fp8_e4m3',
            '--max-model-len', str(MAX_TOKENS),
            '--disable-log-stats',
            '--enable-prefix-caching',
        ]

        self._log_file = open('/kaggle/working/vllm_server.log', 'w')
        self._server_process = subprocess.Popen(
            cmd, stdout=self._log_file, stderr=subprocess.STDOUT,
            start_new_session=True
        )
        self._model_name = 'gpt-oss'

        # Wait for server to be ready
        from openai import OpenAI
        self._client = OpenAI(
            base_url=f'http://0.0.0.0:{port}/v1',
            api_key='sk-local',
            timeout=960,
        )
        print("Waiting for vLLM server to be ready...")
        for i in range(300):
            rc = self._server_process.poll()
            if rc is not None:
                self._log_file.flush()
                with open('/kaggle/working/vllm_server.log') as f:
                    print(f.read()[-2000:])
                raise RuntimeError(f"vLLM server died with code {rc}")
            try:
                self._client.models.list()
                print(f"Server ready after {i+1}s!")
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError("vLLM server failed to start (timeout 300s)")

    def load_api(self, base_url: str, model_name: str):
        """Use an external OpenAI-compatible API (for local testing)."""
        from openai import OpenAI
        self._client = OpenAI(
            base_url=base_url,
            api_key='sk-local',
            timeout=600,
        )
        self._model_name = model_name
        print(f"Using API: {base_url}, model: {model_name}")

    def generate_batch(self, messages_batch: list) -> list:
        """Generate responses for a batch of conversation histories."""
        t0 = time.time()
        n = len(messages_batch)

        # gpt-oss stop token ids
        stop_token_ids = [
            tid for tid in range(200_000, 201_088)
            if tid not in {200005, 200006, 200007, 200008}
        ]

        results = []
        for idx, messages in enumerate(messages_batch):
            if time.time() > CUTOFF_TIME:
                results.append("")
                continue
            call_t0 = time.time()
            try:
                resp = self._client.chat.completions.create(
                    model=self._model_name,
                    messages=messages,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    seed=SEED + idx,
                    extra_body=dict(
                        min_p=MIN_P,
                        stop_token_ids=stop_token_ids,
                        chat_template_kwargs=dict(enable_thinking=True),
                    ),
                )
                choice = resp.choices[0].message
                content = choice.content or ""
                reasoning = getattr(choice, 'reasoning_content', None) or ""
                if reasoning:
                    content = f"<think>{reasoning}</think>\n{content}"
            except Exception as e:
                print(f"    [{idx+1}/{n}] ERROR: {e}")
                content = ""
            dt = time.time() - call_t0
            preview = strip_think(content).replace("\n", " ")[:120]
            print(f"    [{idx+1}/{n}] {dt:.1f}s | {preview}...")
            results.append(content)
        print(f"  Batch done: {n} calls in {time.time()-t0:.1f}s")
        return results

    def predict(self, problem: str) -> int:
        """Solve one problem with TIR + majority voting."""
        if time.time() > CUTOFF_TIME:
            return 0

        msgs_batch = [
            [
                {"role": "system", "content": prompt},
                {"role": "user", "content": problem},
            ]
            for prompt in SYSTEM_PROMPTS[:N_PROMPTS]
        ]

        all_answers = []

        for round_idx in range(MAX_ROUNDS):
            if time.time() > CUTOFF_TIME:
                break

            print(f"  Round {round_idx+1}/{MAX_ROUNDS}: generating {len(msgs_batch)} samples...")
            responses = self.generate_batch(msgs_batch)

            next_batch = []
            for i, resp in enumerate(responses):
                msgs_batch[i].append({"role": "assistant", "content": resp})
                cleaned = strip_think(resp)

                # Extract boxed answers
                boxed = extract_boxed_answers(cleaned)
                if not boxed:
                    boxed = extract_boxed_answers(resp)
                all_answers.extend(boxed)

                # Extract and run Python code
                codes = extract_python_code(cleaned)
                if not codes:
                    codes = extract_python_code(resp)

                code_output_text = ""
                for code in codes:
                    if time.time() > CUTOFF_TIME:
                        break
                    success, output = repl(code)
                    status = "OK" if success else "ERR"
                    print(f"      Code exec [{status}]: {output[:100]}{'...' if len(output)>100 else ''}")
                    if success and output:
                        code_output_text += output + "\n"
                        for num_str in re.findall(r'\b\d{1,5}\b', output):
                            n = int(num_str)
                            if 0 <= n <= 99999:
                                all_answers.append(n)

                if not boxed and code_output_text.strip():
                    msgs_batch[i].append({
                        "role": "user",
                        "content": (
                            f"Code output:\n```\n{code_output_text.strip()}\n```\n"
                            "Continue solving. Put your final answer in \\boxed{}."
                        ),
                    })
                    next_batch.append(msgs_batch[i])
                elif not boxed and round_idx < MAX_ROUNDS - 1:
                    msgs_batch[i].append({
                        "role": "user",
                        "content": "Continue solving. Put your final answer in \\boxed{}.",
                    })
                    next_batch.append(msgs_batch[i])

            msgs_batch = next_batch

            if len(all_answers) >= 5:
                top_count = Counter(all_answers).most_common(1)[0][1]
                if top_count >= 3:
                    break

            if not msgs_batch:
                break

        if msgs_batch and time.time() <= CUTOFF_TIME:
            for msgs in msgs_batch:
                msgs.append({"role": "user", "content": "Output your final answer now as \\boxed{N}."})
            try:
                final_responses = self.generate_batch(msgs_batch)
                for resp in final_responses:
                    cleaned = strip_think(resp)
                    all_answers.extend(extract_boxed_answers(cleaned))
                    all_answers.extend(extract_boxed_answers(resp))
            except Exception:
                pass

        if not all_answers:
            return 0

        answer = select_answer(all_answers)
        print(f"  Votes: {dict(Counter(all_answers))}, Selected: {answer}")
        return answer

    def __del__(self):
        if self._server_process:
            self._server_process.terminate()
            self._server_process.wait()
        if self._log_file:
            self._log_file.close()


model = Model()
print("Model wrapper ready.")

## Cell 8: Load Model

This cell loads the model. On Kaggle it uses vLLM; locally you can skip this and use `model.load_api(...)` instead.

In [ ]:
import subprocess, sys

# Install vLLM 0.15.0 from offline wheels (Python 3.12, CUDA 12.9)
_whl = "/kaggle/input/vllm-wheels-py312-cu129/wheels"
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
    "tensorflow", "keras", "matplotlib", "scikit-learn"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "--no-index", f"--find-links={_whl}", "vllm"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("vLLM installed!")

import vllm
print(f"vLLM {vllm.__version__} OK")

if MODEL_PATH:
    model.load_vllm_server(MODEL_PATH)
else:
    print("No model path — skipping vLLM load.")
    print("For local testing, run: model.load_api('http://localhost:8080/v1', 'model-name')")

## Cell 9: Predict Function (Kaggle API)

This is the function that `AIMO3InferenceServer` calls for each problem.

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    """Kaggle evaluation API entry point."""
    pid = id_.item(0)
    question_text = question.item(0)

    print(f"\n{'='*50}")
    print(f"Problem {pid}")
    print(f"{'='*50}")

    result = model.predict(question_text)
    print(f"Final answer: {result}")

    return pl.DataFrame({"id": pid, "answer": result})

print("predict() function defined.")

## Cell 10: Start Inference Server

This is the main entry point. On Kaggle competition reruns, it starts the inference server. Otherwise it does nothing (you can test locally by calling `model.predict("your problem")` manually).

In [ ]:
if HAS_KAGGLE_EVAL:
    # Kaggle competition mode
    inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        print("Competition rerun detected — starting inference server...")
        inference_server.serve()
    elif os.path.exists("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv"):
        print("Local gateway mode — running against test.csv...")
        inference_server.run_local_gateway(
            ("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",)
        )
    else:
        print("kaggle_evaluation available but no test data. Run manually:")
        print('  model.predict("Find all integers n such that ...")')
else:
    print("Not on Kaggle. To test locally:")
    print('  model.load_api("http://localhost:8080/v1", "model-name")')
    print('  model.predict("Find all integers n such that ...")')